In [1]:
#day6：PyTorch 入门（tensor + shape + 最小 forward）
import torch

In [2]:
# 1. 检查基础版本
print(f"PyTorch version: {torch.__version__}")

# 2. 检查 Apple Silicon GPU (MPS) 是否可用
if torch.backends.mps.is_available():
    print("加速成功：可以使用 M 系列芯片的 GPU 加速 (MPS)！")
else:
    print("目前仅支持 CPU 模式。")

# 3. 测试一个计算任务
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
x = torch.rand(1000, 1000).to(device)
y = torch.mm(x, x)
print("计算测试完成！")

PyTorch version: 2.11.0
目前仅支持 CPU 模式。
计算测试完成！


In [3]:
x = torch.tensor([
    [10.0,2.0, 0.0, 5.0],
    [3.0, 0.0, 1.0, 8.0]
])

In [4]:
x

tensor([[10.,  2.,  0.,  5.],
        [ 3.,  0.,  1.,  8.]])

In [5]:
x.shape

torch.Size([2, 4])

In [6]:
type(x)

torch.Tensor

In [7]:
import torch.nn as nn#导入神经网络模块
model = nn.Linear(4, 2)#定义一个线性层（最简单模型），输入4维，输出2维

In [8]:
y = model(x)#把输入x传入模型，foward前向传播
y

tensor([[ 0.8530,  3.4152],
        [-2.4677,  2.6272]], grad_fn=<AddmmBackward0>)

In [9]:
y.shape

torch.Size([2, 2])

In [10]:
# 查看权重矩阵 W (2x4)，随机初始化
print("当前权重矩阵：")
print(model.weight)

# 查看偏置向量 b (长度 2)
print("\n当前偏置向量：")
print(model.bias)

当前权重矩阵：
Parameter containing:
tensor([[ 0.2870, -0.0622, -0.0317, -0.4681],
        [ 0.2637, -0.1691,  0.1893,  0.1768]], requires_grad=True)

当前偏置向量：
Parameter containing:
tensor([0.4476, 0.2318], requires_grad=True)


In [11]:
#day7：Dataset / DataLoader / batch / inference
import torch

X = torch.tensor([
    [10., 2., 0., 5.],
    [3., 0., 1., 8.],
    [0., 6., 2., 1.],
    [7., 3., 0., 0.],
    [1., 9., 4., 2.]
])

In [12]:
from torch.utils.data import Dataset

class CellDataset(Dataset):#定义一个“数据类”，在pytorch中，数据要封装成 Dataset 才能被 DataLoader 使用
    def __init__(self, X):#初始化，把数据存进去
        self.X = X

    def __len__(self):#返回数据集大小（有多少个细胞）
        return self.X.shape[0]#shape[0]括号中写0是因为X 通常是 (细胞数, 基因数)，shape[0]是告诉pytorch数据集中一共有多少个细胞用于训练

    def __getitem__(self, idx):#随机生成索引，给一个索引 idx，返回一个样本，即每次取一个细胞的表达向量
        return self.X[idx]
    

In [13]:
dataset = CellDataset(X)

In [14]:
len(dataset)

5

In [15]:
dataset[0]

tensor([10.,  2.,  0.,  5.])

In [16]:
dataset[0].shape

torch.Size([4])

In [17]:
from torch.utils.data import DataLoader
loader = DataLoader(dataset, batch_size = 2,shuffle = False)
#把 Dataset 变成“可以一批一批取数据”的对象，false是不打乱顺序

In [18]:
#验证batch
for batch in loader:
    print(batch)
    print(batch.shape)#batch = (batch_size, n_features)

tensor([[10.,  2.,  0.,  5.],
        [ 3.,  0.,  1.,  8.]])
torch.Size([2, 4])
tensor([[0., 6., 2., 1.],
        [7., 3., 0., 0.]])
torch.Size([2, 4])
tensor([[1., 9., 4., 2.]])
torch.Size([1, 4])


In [19]:
import torch.nn as nn
model = nn.Linear(4, 2)
for batch in loader:
    y = model(batch)#inference
    print(y.shape)

torch.Size([2, 2])
torch.Size([2, 2])
torch.Size([1, 2])


In [20]:
#day8:把单细胞数据接入模型
import numpy as np
import pandas as pd
import scanpy as sc

In [21]:
genes = ["geneA", "geneB", "geneC", "geneD"]
cells = ["cell1", "cell2", "cell3", "cell4", "cell5"]

data = np.array([
    [10, 2, 0, 5],
    [3, 0, 1, 8],
    [0, 6, 2, 1],
    [7, 3, 0, 0],
    [1, 9, 4, 2]
])

expr_df = pd.DataFrame(data, index=cells, columns=genes)
expr_df

,geneA,geneB,geneC,geneD
cell1,10,2,0,5
cell2,3,0,1,8
cell3,0,6,2,1
cell4,7,3,0,0
cell5,1,9,4,2


In [22]:
adata = sc.AnnData(expr_df)
adata

AnnData object with n_obs × n_vars = 5 × 4

In [23]:
X_np = adata.X#提取.X表达矩阵

In [24]:
type(X_np)#验证

numpy.ndarray

In [25]:
X_np.shape

(5, 4)

In [26]:
import torch

X_tensor = torch.tensor(X_np, dtype=torch.float32)#把numpy array转换为tensor，指定数据类型为 float32

In [27]:
#验证
X_tensor

tensor([[10.,  2.,  0.,  5.],
        [ 3.,  0.,  1.,  8.],
        [ 0.,  6.,  2.,  1.],
        [ 7.,  3.,  0.,  0.],
        [ 1.,  9.,  4.,  2.]])

In [28]:
X_tensor.shape

torch.Size([5, 4])

In [29]:
import torch.nn as nn

model = nn.Linear(X_tensor.shape[1], 2)#定义模型（自动适配维度），1是列，基因数

In [30]:
y = model(X_tensor)#跑forward，输入（5，4），输出（5，2）
y.shape

torch.Size([5, 2])

In [31]:
#day9:读懂一个模型的 forward
import torch
import torch.nn as nn

class SimpleModel(nn.Module):#定义一个模型类
    def __init__(self):#初始化模型结构
        super().__init__()#调用父类（nn.Module）的初始化，固定写法
        self.fc1 = nn.Linear(4, 3)#第一层线性层
        self.relu = nn.ReLU()#第二层激活函数，非线性
        self.fc2 = nn.Linear(3, 2)#第三层线性层

    def forward(self, x):#定义数据怎么流动
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

In [32]:
#运行模型
model = SimpleModel()

x = torch.tensor([
    [10., 2., 0., 5.],
    [3., 0., 1., 8.]
])

y = model(x)
y

tensor([[4.0334, 2.1955],
        [2.2282, 1.7384]], grad_fn=<AddmmBackward0>)

In [33]:
y.shape

torch.Size([2, 2])